#  RAG 성능 평가 📊

---


## 1. 개요 및 환경 설정

### 1.1 RAG 평가가 중요한 이유

**RAG(Retrieval-Augmented Generation)** 시스템은 외부 지식을 검색하여 LLM의 응답 품질을 향상시키는 기술입니다. 하지만 RAG 시스템의 성능을 객관적으로 측정하고 개선하기 위해서는 체계적인 평가가 필요합니다.

#### 주요 평가 목표:
- **검색 품질**: 관련성 높은 문서를 얼마나 잘 찾는가?
- **생성 품질**: 검색된 정보를 얼마나 잘 활용하는가?
- **종합 성능**: 사용자에게 얼마나 유용한 답변을 제공하는가?

<center>
<img src="https://raw.githubusercontent.com/tsdata/image_files/main/202505/rag_evaluation.png" alt="rag" align="center" border="0"  width="1000" height=auto>
</center>

[출처] https://arxiv.org/abs/2405.07437

### 1.2 환경 설정

In [7]:
# 환경변수 설정
from dotenv import load_dotenv
import os
load_dotenv()

# 기본 라이브러리
import json
import pandas as pd
import numpy as np
from pprint import pprint
from glob import glob

# LangSmith 추적 설정 (선택사항)
print(f"LangSmith 추적: {os.getenv('LANGSMITH_TRACING')}")

LangSmith 추적: true


---

## 2. RAG 평가의 핵심 개념

### 2.1 평가 차원 (Evaluation Dimensions)

<div align="center">

| 평가 영역 | 세부 지표 | 설명 |
|-----------|-----------|------|
| **검색 (Retrieval)** | Context Relevancy | 검색된 문서가 질문과 얼마나 관련있는가? |
|  | Context Recall | 정답에 필요한 모든 정보가 검색되었는가? |
| **생성 (Generation)** | Faithfulness | 생성된 답변이 검색된 문서에 충실한가? |
|  | Answer Relevancy | 생성된 답변이 질문과 관련있는가? |
| **종합** | Answer Correctness | 생성된 답변이 정답과 일치하는가? |

</div>

### 2.2 평가 방법론

#### A. Reference-Free 평가
- **장점**: 정답 데이터 없이도 평가 가능
- **방법**: LLM-as-Judge 방식 활용
- **도구**: RAGAS, LangSmith 등

#### B. Reference-Based 평가
- **장점**: 객관적이고 일관된 평가
- **방법**: 정답과 비교하여 평가
- **지표**: BLEU, ROUGE, Semantic Similarity 등

---

## 3. RAGAS 프레임워크 소개

### 3.1 RAGAS란?

**RAGAS (Retrieval-Augmented Generation Assessment)** 는 RAG 시스템을 위한 오픈소스 평가 프레임워크입니다.

#### 주요 특징:
- ✅ **Reference-Free**: 정답 없이도 평가 가능
- ✅ **LLM-as-Judge**: GPT-4 등을 활용한 자동 평가
- ✅ **구성요소별 평가**: 검색과 생성을 개별적으로 평가
- ✅ **LangChain 통합**: 기존 RAG 파이프라인과 쉽게 연동

### 3.2 RAGAS 핵심 지표

```python
from ragas.metrics import (
    context_relevancy,      # 컨텍스트 관련성
    context_recall,         # 컨텍스트 회상률
    faithfulness,          # 충실도
    answer_relevancy,      # 답변 관련성
    answer_correctness     # 답변 정확성
)
```

#### 지표별 상세 설명:

1. **Context Relevancy (컨텍스트 관련성)**
   - 검색된 문서가 질문과 얼마나 관련있는지 측정
   - 계산식: `관련 문장 수 / 전체 문장 수`

2. **Context Recall (컨텍스트 검출률)**
   - 정답 생성에 필요한 정보가 얼마나 검색되었는지 측정
   - 정답(ground truth) 필요

3. **Faithfulness (충실도)**
   - 생성된 답변이 검색된 문서에 얼마나 충실한지 측정
   - 환각(hallucination) 검출에 중요

4. **Answer Relevancy (답변 관련성)**
   - 생성된 답변이 질문과 얼마나 관련있는지 측정

5. **Answer Correctness (답변 정확성)**
   - 생성된 답변이 정답과 얼마나 일치하는지 측정

---

## 4. 기본 RAG 시스템 구축

### 4.1 문서 준비 및 처리

In [8]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_documents(file_paths):
    """텍스트 파일들을 로드하는 함수"""
    documents = []
    for path in file_paths:
        try:
            loader = TextLoader(path, encoding='utf-8')
            documents.extend(loader.load())
        except Exception as e:
            print(f"파일 로드 실패 {path}: {e}")
    return documents

# 예시: 한국어 문서 로드
korean_files = glob('./data/*_KR.md')
documents = load_documents(korean_files)

print(f"로드된 문서 수: {len(documents)}")
if documents:
    print(f"첫 번째 문서 미리보기:\n{documents[0].page_content[:200]}...")

로드된 문서 수: 2
첫 번째 문서 미리보기:
Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla...


### 4.2 문서 분할 (Text Splitting)


In [9]:
# 한국어 텍스트에 최적화된 분할기 설정
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    separators=['\n\n', '\n', r'(?<=[.!?])\s+'],  # 문장 단위 분할
    chunk_size=300,           # 청크 크기
    chunk_overlap=0,         # 중복 영역
    is_separator_regex=True,  # 정규식 사용
    keep_separator=True       # 구분자 유지
)

# 문서 분할 실행
split_docs = text_splitter.split_documents(documents)

print(f"분할된 청크 수: {len(split_docs)}")
print(f"\n첫 번째 청크:")
print(f"메타데이터: {split_docs[0].metadata}")
print(f"내용: {split_docs[0].page_content}")

분할된 청크 수: 39

첫 번째 청크:
메타데이터: {'source': './data/테슬라_KR.md'}
내용: Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.


### 4.3 벡터 스토어 생성

In [14]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 임베딩 모델 초기화
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536  # 선택사항: 차원 축소로 성능 향상
)

# Chroma 벡터 스토어 생성
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    collection_name="rag_evaluation_demo",
    persist_directory="./local_chroma_db",
    collection_metadata={'hnsw:space': 'cosine'}  # 코사인 유사도 사용
)

print(f"벡터 스토어에 저장된 문서 수: {vector_store._collection.count()}")

벡터 스토어에 저장된 문서 수: 39


### 4.4 RAG 체인 구성


In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM 초기화
llm = ChatOpenAI(
    model="gpt-4.1-mini",  # 비용 효율적인 모델
    temperature=0,        # 일관된 결과를 위해 0으로 설정
)

# 검색기 설정
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}  # 상위 5개 문서 검색
)

# 프롬프트 템플릿
prompt_template = ChatPromptTemplate.from_template("""
다음 컨텍스트를 바탕으로 질문에 답하세요. 
컨텍스트에 없는 정보는 추측하지 마세요.

컨텍스트:
{context}

질문: {question}

답변:
""")

# RAG 체인 구성
def format_docs(docs):
    """문서 리스트를 문자열로 변환"""
    return "\n\n".join(doc.page_content for doc in docs)

def rag_chain(question: str) -> dict:
    """RAG 체인 실행 함수"""

    # 문서 검색
    retrieved_docs = retriever.invoke(question)
    
    # 컨텍스트 준비
    context = format_docs(retrieved_docs)
    
    # LLM으로 답변 생성
    response = llm.invoke(
        prompt_template.format_prompt(
            context=context, 
            question=question
        )
    ).content
    
    return {
        "question": question,
        "context": context,
        "answer": response,
        "retrieved_docs": retrieved_docs
    }

# 테스트 실행
test_question = "리비안의 설립자는 누구인가요?"
result = rag_chain(test_question)

print(f"질문: {result['question']}")
print(f"답변: {result['answer']}")
print(f"검색된 문서 수: {len(result['retrieved_docs'])}")

질문: 리비안의 설립자는 누구인가요?
답변: 리비안의 설립자는 R. J. 스캐린지입니다.
검색된 문서 수: 5


---

## 5. 평가 데이터셋 생성

### 5.1 RAGAS 합성 데이터

- uv add rangas==0.3.1
- uv add rapidfuzz

In [17]:
from ragas.testset import TestsetGenerator
from ragas.testset.persona import Persona
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# LLM과 임베딩 래퍼 설정
generator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4.1", temperature=0.2)
)
generator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

# 한국어 페르소나 정의 
personas = [
    Persona(
        name="graduate_researcher",
        role_description="""미국 전기차 시장을 연구하는 한국인 박사과정 연구원입니다. 
        전기차 정책, 시장 동향, 기술적 세부사항에 대해 깊이 있는 분석적 질문을 합니다. 
        학술적 용어를 사용하며 데이터와 근거를 중요하게 생각합니다. 한국어만 사용합니다.""",
    ),
    Persona(
        name="masters_student",
        role_description="""전기차 산업을 공부하는 한국인 석사과정 학생입니다. 
        미국 전기차 시장의 기초 개념과 트렌드를 이해하려 노력하며, 
        명확하고 이해하기 쉬운 설명을 선호합니다. 한국어만 사용합니다.""",
    ),
    Persona(
        name="industry_analyst",
        role_description="""한국 자동차 회사에서 미국 전기차 시장을 분석하는 주니어 연구원입니다. 
        실무적인 시장 데이터, 경쟁사 동향, 비즈니스 인사이트에 관심이 많으며, 
        실행 가능한 정보를 중요하게 생각합니다. 한국어만 사용합니다.""",
    ),
    Persona(
        name="policy_researcher",
        role_description="""한국 정부기관에서 전기차 정책을 연구하는 연구원입니다. 
        미국의 전기차 관련 정책, 인센티브, 규제에 대해 관심이 많으며, 
        한국 정책에 적용 가능한 시사점을 찾고 있습니다. 한국어만 사용합니다.""",
    )
]


In [18]:
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

# 기본 프롬프트 (영어 버전) 확인
synthesizer = SingleHopSpecificQuerySynthesizer(llm=generator_llm)

synthesizer.get_prompts()

{'query_answer_generation_prompt': QueryAnswerGenerationPrompt(instruction=Generate a single-hop query and answer based on the specified conditions (persona, term, style, length) and the provided context. Ensure the answer is entirely faithful to the context, using only the information directly from the provided context.### Instructions:
 1. **Generate a Query**: Based on the context, persona, term, style, and length, create a question that aligns with the persona's perspective and incorporates the term.
 2. **Generate an Answer**: Using only the content from the provided context, construct a detailed answer to the query. Do not add any information not included in or inferable from the context.
 , examples=[(QueryCondition(persona=Persona(name='Software Engineer', role_description='Focuses on coding best practices and system design.'), term='microservices', query_style='Formal', query_length='Medium', context='Microservices are an architectural style where applications are structured a

In [19]:
# 한국어 프롬프트로 변환
korean_prompts = await synthesizer.adapt_prompts(
    language="korean", 
    llm=generator_llm
)
synthesizer.set_prompts(**korean_prompts)

In [20]:
print(korean_prompts['query_answer_generation_prompt'].instruction)

Generate a single-hop query and answer based on the specified conditions (persona, term, style, length) and the provided context. Ensure the answer is entirely faithful to the context, using only the information directly from the provided context.### Instructions:
1. **Generate a Query**: Based on the context, persona, term, style, and length, create a question that aligns with the persona's perspective and incorporates the term.
2. **Generate an Answer**: Using only the content from the provided context, construct a detailed answer to the query. Do not add any information not included in or inferable from the context.



In [21]:
synthesizer.get_prompts()

{'query_answer_generation_prompt': QueryAnswerGenerationPrompt(instruction=Generate a single-hop query and answer based on the specified conditions (persona, term, style, length) and the provided context. Ensure the answer is entirely faithful to the context, using only the information directly from the provided context.### Instructions:
 1. **Generate a Query**: Based on the context, persona, term, style, and length, create a question that aligns with the persona's perspective and incorporates the term.
 2. **Generate an Answer**: Using only the content from the provided context, construct a detailed answer to the query. Do not add any information not included in or inferable from the context.
 , examples=[(QueryCondition(persona=Persona(name='소프트웨어 엔지니어', role_description='코딩 모범 사례와 시스템 설계에 중점을 둡니다.'), term='마이크로서비스', query_style='격식체', query_length='중간', context='마이크로서비스는 애플리케이션을 느슨하게 결합된 서비스들의 집합으로 구성하는 아키텍처 스타일입니다. 각 서비스는 세분화되어 있으며 하나의 기능에 집중합니다.'), GeneratedQueryAnswer(query='소프트

In [22]:
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.specific import MultiHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.abstract import MultiHopAbstractQuerySynthesizer


async def create_korean_query_distribution():
    """한국어 최적화된 Query Distribution 생성"""
    
    # Query Synthesizer들을 한국어로 적응
    synthesizers = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.6),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.2),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.2)
    ]
    
    # 각 synthesizer의 프롬프트를 한국어로 적응
    korean_query_distribution = []
    for synthesizer, weight in synthesizers:
        try:
            # 한국어 프롬프트 적응
            korean_prompts = await synthesizer.adapt_prompts(
                language="korean", 
                llm=generator_llm
            )
            synthesizer.set_prompts(**korean_prompts)
            print(f"{synthesizer.__class__.__name__} 한국어 적응 완료")
            korean_query_distribution.append((synthesizer, weight))
        except Exception as e:
            print(f"{synthesizer.__class__.__name__} 한국어 적응 실패: {e}")
            # 적응에 실패해도 원본 synthesizer는 포함
            korean_query_distribution.append((synthesizer, weight))
    
    return korean_query_distribution

In [24]:
# 합성 데이터셋 생성
async def generate_korean_testset(split_docs, testset_size=60):
    """한국어 테스트셋 생성"""
    
    # 한국어 적응된 Query Distribution 생성
    korean_query_distribution = await create_korean_query_distribution()
    
    # TestsetGenerator 생성
    testset_generator = TestsetGenerator(
        llm=generator_llm,
        embedding_model=generator_embeddings,
        persona_list=personas
    )
    
    # 테스트셋 생성 (query_distribution 매개변수 사용)
    synthetic_dataset = testset_generator.generate_with_langchain_docs(
        documents=split_docs,
        testset_size=testset_size,
        query_distribution=korean_query_distribution  # 올바른 매개변수
    )
    
    return synthetic_dataset


synthetic_dataset = await generate_korean_testset(split_docs, testset_size=50)

SingleHopSpecificQuerySynthesizer 한국어 적응 완료
MultiHopSpecificQuerySynthesizer 한국어 적응 완료
MultiHopAbstractQuerySynthesizer 한국어 적응 완료


Applying CustomNodeFilter:   0%|          | 0/39 [00:00<?, ?it/s]         Node 392079ae-a0a6-4f51-932a-62db6d66b041 does not have a summary. Skipping filtering.
Node 16e9d2ee-a62f-4140-8470-fa1bfc5386d0 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  21%|██        | 8/39 [00:00<00:02, 11.96it/s]Node 779aefb9-e703-4765-b59c-5f482368d9e3 does not have a summary. Skipping filtering.
Node 593f7478-cbf0-412f-bf73-4f69f14b1b75 does not have a summary. Skipping filtering.
Node 2e896d7f-929b-47e4-ae14-6c0cd6bf4541 does not have a summary. Skipping filtering.
Generating Samples: 100%|██████████| 50/50 [00:09<00:00,  5.26it/s]


In [25]:
# 결과 확인
df = synthetic_dataset.to_pandas()
print(f"생성된 합성 데이터셋 크기: {len(df)}")
print(f"\n컬럼: {list(df.columns)}")
print(f"\n첫 번째 샘플:")
print(f"질문: {df.iloc[0]['user_input']}")
print(f"정답: {df.iloc[0]['reference']}")

# CSV로 저장
df.to_csv('./data/synthetic_testset.csv', index=False, encoding='utf-8')

생성된 합성 데이터셋 크기: 50

컬럼: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

첫 번째 샘플:
질문: Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그리고 회사 설립 과정에서 어떤 의미를 가지는지 자세히 설명해 주세요.
정답: Tesla, Inc.는 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립하였으며, Nikola Tesla를 기리기 위해 회사 이름을 지었습니다. Nikola Tesla는 회사의 명칭에 직접적으로 연결되어 있으며, 설립자들이 그의 업적을 기념하고자 Tesla라는 이름을 선택한 것입니다.


In [26]:
df

,user_input,reference_contexts,reference,synthesizer_name
0,"Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그...","[Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사...","Tesla, Inc.는 2003년 7월 Martin Eberhard와 Marc Ta...",single_hop_specifc_query_synthesizer
1,Tesla가 전기차 시장에서 차지하는 위치와 주요 성과에 대해 설명해 주실 수 있나요?,[Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (2...,"Tesla는 2020년 7월 이후 세계에서 가장 가치 있는 자동차 제조업체이며, 2...",single_hop_specifc_query_synthesizer
2,Martin Eberhard는 Tesla의 설립 과정에서 어떤 역할을 했습니까?,"[Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Mu...","Martin Eberhard는 2003년 7월 1일에 Tesla Motors, In...",single_hop_specifc_query_synthesizer
3,Michael Marks 뭐 했어요?,[Elon Musk는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략...,Michael Marks는 Ze'ev Drori가 인수하기 전에 임시 CEO를 역임...,single_hop_specifc_query_synthesizer
4,테슬라 NASDAQ 뭐임?,[Roadster 생산은 2008년에 시작되었습니다. 2009년 1월까지 Tesla...,"테슬라는 2010년 6월 NASDAQ에 상장해서 2억 2,600만 달러를 조달했습니다.",single_hop_specifc_query_synthesizer
5,미국 전기차 시장에서 Tesla의 Model S가 출시 이후 어떤 성과를 거두었는지...,[Tesla는 2012년 6월 Model S 고급 세단을 출시했습니다. Model ...,Tesla는 2012년 6월 Model S 고급 세단을 출시했습니다. Model S...,single_hop_specifc_query_synthesizer
6,Tesla의 Model Y가 글로벌 확장 과정에서 어떤 역할을 했는지 설명해 주시겠...,[Tesla는 2016년 11월 SolarCity를 26억 달러에 인수하여 Tesl...,Model Y는 2019년 이후 Tesla의 글로벌 확장 과정에서 중요한 역할을 했...,single_hop_specifc_query_synthesizer
7,Tesla가 2020년에 생산 능력 확장을 위해 건설한 Gigafactory Tex...,[2019년 7월부터 2020년 6월까지 Tesla는 4분기 연속 흑자를 보고했으며...,Tesla는 2020년에 생산 능력 확장을 위해 Gigafactory Texas를 ...,single_hop_specifc_query_synthesizer
8,2020년 3월에 미국 전기차 기업 Tesla가 어떤 중요한 사건을 겪었는지 자세히...,"[2023년 3월, Tesla는 잠재적인 관세 영향으로 인해 지연된 Gigafact...","2020년 3월, Tesla는 COVID-19 팬데믹 초기에 프리몬트 공장을 폐쇄했...",single_hop_specifc_query_synthesizer
9,미국 전기차 시장에서 테슬라가 델라웨어와 관련하여 어떤 중요한 결정을 내렸는지 자세...,"[2021년 초, Tesla는 Bitcoin에 15억 달러를 투자하고 환경 문제로 ...","2024년 6월, 테슬라는 법인 설립지를 델라웨어에서 텍사스로 이전하는 결정을 내렸...",single_hop_specifc_query_synthesizer


### 5.2 평가용 데이터셋 준비

- RAG 시스템의 출력을 평가 데이터셋에 추가

In [27]:
from ragas import EvaluationDataset

def create_evaluation_dataset(questions_data, rag_chain_func):
    """평가용 데이터셋을 생성하는 함수"""
    
    evaluation_data = []
    
    # 각 질문에 대해 RAG 체인을 실행하고 평가 데이터 구성
    for item in questions_data:
        if "question" in item: # 수동 데이터셋의 경우
            question = item["question"]
            ground_truth = item.get("ground_truth", "")
        else: # 합성 데이터셋의 경우
            question = item["user_input"]
            ground_truth = item["reference"]

        # RAG 체인 실행
        result = rag_chain_func(question)
        
        # 평가 데이터 구성
        eval_sample = {
            "user_input": question,
            "response": result["answer"],
            "retrieved_contexts": [doc.page_content for doc in result["retrieved_docs"]],
            "reference": ground_truth
        }
        
        evaluation_data.append(eval_sample)
        
        print(f"처리 완료: {question[:50]}...")
    
    return EvaluationDataset.from_list(evaluation_data)

# 평가 데이터셋 준비 
evaluation_questions = synthetic_dataset.to_list()  # 합성 데이터셋을 평가 질문으로 사용

print(f"총 {len(evaluation_questions)}개의 평가 질문이 있습니다.")

총 50개의 평가 질문이 있습니다.


In [28]:
# 평가 데이터셋 생성
eval_dataset = create_evaluation_dataset(evaluation_questions, rag_chain) 

# 데이터프레임으로 확인
eval_df = eval_dataset.to_pandas()
print(f"\n평가 데이터셋 생성 완료:")
print(f"- 샘플 수: {len(eval_df)}")
print(f"- 컬럼: {list(eval_df.columns)}")

# 저장
eval_df.to_csv('./data/evaluation_dataset.csv', index=False, encoding='utf-8')

처리 완료: Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그리고 회...
처리 완료: Tesla가 전기차 시장에서 차지하는 위치와 주요 성과에 대해 설명해 주실 수 있나요?...
처리 완료: Martin Eberhard는 Tesla의 설립 과정에서 어떤 역할을 했습니까?...
처리 완료: Michael Marks 뭐 했어요?...
처리 완료: 테슬라 NASDAQ 뭐임?...
처리 완료: 미국 전기차 시장에서 Tesla의 Model S가 출시 이후 어떤 성과를 거두었는지, 그리...
처리 완료: Tesla의 Model Y가 글로벌 확장 과정에서 어떤 역할을 했는지 설명해 주시겠습니까?...
처리 완료: Tesla가 2020년에 생산 능력 확장을 위해 건설한 Gigafactory Texas에 ...
처리 완료: 2020년 3월에 미국 전기차 기업 Tesla가 어떤 중요한 사건을 겪었는지 자세히 설명해...
처리 완료: 미국 전기차 시장에서 테슬라가 델라웨어와 관련하여 어떤 중요한 결정을 내렸는지 자세히 설명...
처리 완료: 2024년 11월 테슬라 자동차 모델 뭐있나요?...
처리 완료: 테슬라 Roadster는 언제 출시되었고 몇 인승인가요?...
처리 완료: 미국 전기차 시장에서 Model S의 주요 특징과 출시 시점에 대해 설명해 주시겠습니까?...
처리 완료: 2016년에 테슬라 Model 3가 처음 공개된 이후, 해당 연도의 공개와 관련된 주요 특...
처리 완료: Tesla, Inc.가 출시한 Tesla Semi의 주요 기술적 특징과 초기 배송 현황에 ...
처리 완료: 2019년 11월에 발표된 전기차 모델에 대해 알려주세요....
처리 완료: Tesla 차세대 차량은 어떤 특징을 가지고 있으며 언제 출시될 예정인가요?...
처리 완료: 표준 연결 뭐임?...
처리 완료: 미국 테슬라 차체 수리점 어디에 있나? 테슬라 차체 수리점 역할 뭐임?...
처리 완료: 미국 전기차 시장에서 Tesla Insurance Se

In [29]:
eval_df

,user_input,retrieved_contexts,response,reference
0,"Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그...","[Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사...","Nikola Tesla는 Tesla, Inc.의 이름에 직접적인 영감을 준 인물입니...","Tesla, Inc.는 2003년 7월 Martin Eberhard와 Marc Ta..."
1,Tesla가 전기차 시장에서 차지하는 위치와 주요 성과에 대해 설명해 주실 수 있나요?,[Tesla는 원격 진단 및 수리를 우선시하여 모바일 기술자를 파견하거나 필요한 경...,Tesla는 전기차 시장에서 선도적인 위치를 차지하고 있는 미국의 다국적 자동차 및...,"Tesla는 2020년 7월 이후 세계에서 가장 가치 있는 자동차 제조업체이며, 2..."
2,Martin Eberhard는 Tesla의 설립 과정에서 어떤 역할을 했습니까?,"[Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사...",Martin Eberhard는 2003년 7월 Tesla Motors를 Marc T...,"Martin Eberhard는 2003년 7월 1일에 Tesla Motors, In..."
3,Michael Marks 뭐 했어요?,[Elon Musk는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략...,Michael Marks는 임시 CEO를 역임했습니다.,Michael Marks는 Ze'ev Drori가 인수하기 전에 임시 CEO를 역임...
4,테슬라 NASDAQ 뭐임?,[- **회사 유형:** 상장\n- **거래소:** NASDAQ: RIVN\n- *...,테슬라의 NASDAQ 코드는 컨텍스트에 명시되어 있지 않습니다.,"테슬라는 2010년 6월 NASDAQ에 상장해서 2억 2,600만 달러를 조달했습니다."
5,미국 전기차 시장에서 Tesla의 Model S가 출시 이후 어떤 성과를 거두었는지...,[Tesla는 2012년 6월 Model S 고급 세단을 출시했습니다. Model ...,"Tesla의 Model S는 2012년 6월에 출시된 고급 전기 세단으로, 출시 이...",Tesla는 2012년 6월 Model S 고급 세단을 출시했습니다. Model S...
6,Tesla의 Model Y가 글로벌 확장 과정에서 어떤 역할을 했는지 설명해 주시겠...,[Tesla는 2016년 11월 SolarCity를 26억 달러에 인수하여 Tesl...,Tesla의 Model Y는 2019년 3월에 처음 공개되어 2020년 3월부터 배...,Model Y는 2019년 이후 Tesla의 글로벌 확장 과정에서 중요한 역할을 했...
7,Tesla가 2020년에 생산 능력 확장을 위해 건설한 Gigafactory Tex...,"[2023년 3월, Tesla는 잠재적인 관세 영향으로 인해 지연된 Gigafact...",Gigafactory Texas는 2020년에 건설된 Tesla의 주요 생산 시설로...,Tesla는 2020년에 생산 능력 확장을 위해 Gigafactory Texas를 ...
8,2020년 3월에 미국 전기차 기업 Tesla가 어떤 중요한 사건을 겪었는지 자세히...,[Tesla는 2016년 11월 SolarCity를 26억 달러에 인수하여 Tesl...,2020년 3월에 미국 전기차 기업 Tesla는 Model Y 중형 크로스오버 SU...,"2020년 3월, Tesla는 COVID-19 팬데믹 초기에 프리몬트 공장을 폐쇄했..."
9,미국 전기차 시장에서 테슬라가 델라웨어와 관련하여 어떤 중요한 결정을 내렸는지 자세...,"[2024년 12월, 델라웨어 법원은 부적절한 이사회 승인을 이유로 Elon Mus...","2024년 6월, Tesla는 법인 설립지를 델라웨어에서 텍사스로 이전하는 중요한 ...","2024년 6월, 테슬라는 법인 설립지를 델라웨어에서 텍사스로 이전하는 결정을 내렸..."


---

## 6. 실습 3: RAGAS 평가 수행

### 6.1 기본 평가 실행

In [30]:
from ragas import evaluate
from ragas.metrics import ( #평가지표들
    LLMContextRecall,
    Faithfulness, 
    AnswerRelevancy,
    ContextPrecision,
    FactualCorrectness
)

# 평가용 LLM 설정
evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4.1-mini", temperature=0)
)

# 평가 지표 선택
metrics = [
    LLMContextRecall(llm=evaluator_llm),           # 컨텍스트 검출율
    Faithfulness(llm=evaluator_llm),               # 충실도
    AnswerRelevancy(llm=evaluator_llm),            # 답변 관련성
    ContextPrecision(llm=evaluator_llm),           # 컨텍스트 정밀도
    FactualCorrectness(llm=evaluator_llm)          # 사실적 정확성
]

print("RAGAS 평가 시작...")

# 평가 실행
results = evaluate(
    dataset=eval_dataset[:10],  # 처음 10개 샘플로 평가 (테스트 목적)
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=generator_embeddings
)

print("평가 완료!")
print(f"\n전체 평가 결과:")
print(results)

RAGAS 평가 시작...


Evaluating: 100%|██████████| 50/50 [00:42<00:00,  1.17it/s]


평가 완료!

전체 평가 결과:
{'context_recall': 0.8000, 'faithfulness': 0.7699, 'answer_relevancy': 0.4587, 'context_precision': 0.5867, 'factual_correctness(mode=f1)': 0.3400}


### 6.2 상세 결과 분석


In [31]:
# 결과를 DataFrame으로 변환
results_df = results.to_pandas()

print(f"\n상세 평가 결과:")
print("="*80)

# 각 샘플별 상세 결과
for idx, row in results_df.iterrows():
    print(f"\n[샘플 {idx+1}]")
    print(f"질문: {row['user_input'][:100]}...")
    print(f"답변: {row['response'][:100]}...")
    
    # 지표별 점수 출력
    for col in results_df.columns:
        if col not in ['user_input', 'response', 'retrieved_contexts', 'reference']:
            if pd.notna(row[col]):
                print(f"  {col}: {row[col]:.3f}")
    print("-" * 50)


상세 평가 결과:

[샘플 1]
질문: Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그리고 회사 설립 과정에서 어떤 의미를 가지는지 자세히 설명해 주세요....
답변: Nikola Tesla는 Tesla, Inc.의 이름에 직접적인 영감을 준 인물입니다. Tesla, Inc.는 2003년 7월 Martin Eberhard와 Marc Tarpenn...
  context_recall: 1.000
  faithfulness: 0.375
  answer_relevancy: 0.730
  context_precision: 1.000
  factual_correctness(mode=f1): 0.550
--------------------------------------------------

[샘플 2]
질문: Tesla가 전기차 시장에서 차지하는 위치와 주요 성과에 대해 설명해 주실 수 있나요?...
답변: Tesla는 전기차 시장에서 선도적인 위치를 차지하고 있는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 2003년에 설립되어 Nikola Tesla의 이름을 따왔으며, Elo...
  context_recall: 0.000
  faithfulness: 0.842
  answer_relevancy: 0.556
  context_precision: 0.000
  factual_correctness(mode=f1): 0.150
--------------------------------------------------

[샘플 3]
질문: Martin Eberhard는 Tesla의 설립 과정에서 어떤 역할을 했습니까?...
답변: Martin Eberhard는 2003년 7월 Tesla Motors를 Marc Tarpenning과 함께 설립했으며, 초대 CEO를 역임했습니다....
  context_recall: 1.000
  faithfulness: 1.000
  answer_relevancy: 0.564
  context_p

In [32]:
# 통계 요약
print(f"\n📊 평가 지표 통계:")
print("="*50)

numeric_columns = results_df.select_dtypes(include=[np.number]).columns 
summary_stats = results_df[numeric_columns].describe()

for metric in numeric_columns:
    mean_score = summary_stats.loc['mean', metric]
    std_score = summary_stats.loc['std', metric]
    print(f"{metric}:")
    print(f"  평균: {mean_score:.3f} (±{std_score:.3f})")
    print(f"  범위: {summary_stats.loc['min', metric]:.3f} ~ {summary_stats.loc['max', metric]:.3f}")



📊 평가 지표 통계:
context_recall:
  평균: 0.800 (±0.422)
  범위: 0.000 ~ 1.000
faithfulness:
  평균: 0.770 (±0.245)
  범위: 0.375 ~ 1.000
answer_relevancy:
  평균: 0.459 (±0.241)
  범위: 0.000 ~ 0.730
context_precision:
  평균: 0.587 (±0.438)
  범위: 0.000 ~ 1.000
factual_correctness(mode=f1):
  평균: 0.340 (±0.249)
  범위: 0.000 ~ 0.670


## 🎯 메트릭별 핵심 평가 요소

| 메트릭 | 평가 대상 | 핵심 질문 | 정의 |
|--------|-----------|-----------|------|
| **LLMContextRecall** | 검색 품질 | "필요한 정보가 잘 검색되었는가?" | 참조 답변(reference)의 주장들이 검색된 컨텍스트에 의해 얼마나 잘 지원되는지를 측정하는 지표 |
| **Faithfulness** | 생성 품질 | "답변이 컨텍스트에 충실한가?" | 생성된 답변이 검색된 컨텍스트에 얼마나 충실한지(일치하는지) 측정하는 지표 |
| **AnswerRelevancy** | 답변 품질 | "답변이 질문과 관련이 있는가?" | 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지 측정하는 지표 |
| **ContextPrecision** | 검색 정밀도 | "검색된 것들이 실제로 유용한가?" | 검색된 컨텍스트 중 질문과 실제로 관련된 것들의 비율을 측정하는 지표 |
| **FactualCorrectness** | 사실 정확성 | "답변이 사실적으로 정확한가?" | 생성된 응답과 참조 답변 간의 사실적 정확성을 측정하는 지표 |

In [33]:
# 결과 저장
results_df.to_csv('./data/ragas_evaluation_results.csv', index=False, encoding='utf-8')

---

## 7. A/B 테스트를 통한 시스템 비교

In [34]:
def compare_rag_systems(eval_questions, system_a, system_b, system_names=["System A", "System B"]):
    """두 RAG 시스템을 비교 평가하는 함수"""
    print(f"🔬 A/B 테스트: {system_names[0]} vs {system_names[1]}")
    print("="*60)
    
    # 각 시스템의 결과 생성
    results_a = create_evaluation_dataset(eval_questions, system_a)
    results_b = create_evaluation_dataset(eval_questions, system_b)
    
    # 평가 실행
    eval_a = evaluate(results_a, metrics=metrics, llm=evaluator_llm, embeddings=generator_embeddings)
    eval_b = evaluate(results_b, metrics=metrics, llm=evaluator_llm, embeddings=generator_embeddings)
    
    eval_a_dict = eval_a.to_pandas().mean(numeric_only=True).to_dict()
    eval_b_dict = eval_b.to_pandas().mean(numeric_only=True).to_dict()
    
    # 결과 비교
    comparison_results = {}
    for metric in eval_a_dict.keys():
        if isinstance(eval_a_dict[metric], (int, float)) and isinstance(eval_b_dict[metric], (int, float)):
            score_a = eval_a_dict[metric]
            score_b = eval_b_dict[metric]
            improvement = ((score_b - score_a) / score_a) * 100 if score_a != 0 else 0
            
            comparison_results[metric] = {
                system_names[0]: score_a,
                system_names[1]: score_b,
                'improvement_%': improvement
            }
            
            winner = system_names[1] if score_b > score_a else system_names[0]
            print(f"{metric}:")
            print(f"  {system_names[0]}: {score_a:.3f}")
            print(f"  {system_names[1]}: {score_b:.3f}")
            print(f"  승자: {winner} (개선: {improvement:+.1f}%)")
            print()
    
    return comparison_results

In [35]:
# 다른 검색 파라미터를 가진 시스템 비교
def rag_chain_v2(question: str) -> dict:
    """개선된 RAG 체인 (mmr 문서 검색)"""
    retriever_v2 = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k":10, "lambda_mult": 0.5}
    )
    retrieved_docs = retriever_v2.invoke(question)
    context = format_docs(retrieved_docs)
    
    # 더 상세한 프롬프트
    detailed_prompt = """당신은 도움이 되는 AI 어시스턴트입니다. 
주어진 컨텍스트를 신중하게 분석하여 정확하고 유용한 답변을 제공하세요.

컨텍스트:
{context}

질문: {question}

지침:
1. 컨텍스트에 기반하여 답변하세요
2. 확실하지 않은 정보는 추측하지 마세요
3. 답변은 명확하고 구체적으로 작성하세요

답변:"""
    
    response = llm.invoke(
        detailed_prompt.format(context=context, question=question)
    ).content
    
    return {
        "question": question,
        "context": context,
        "answer": response,
        "retrieved_docs": retrieved_docs
    }

# 비교 실행
comparison = compare_rag_systems(
    evaluation_questions[:3],  # 샘플 3개만 사용
    rag_chain, 
    rag_chain_v2,
    ["기본 시스템", "개선된 시스템"]
)

🔬 A/B 테스트: 기본 시스템 vs 개선된 시스템
처리 완료: Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그리고 회...
처리 완료: Tesla가 전기차 시장에서 차지하는 위치와 주요 성과에 대해 설명해 주실 수 있나요?...
처리 완료: Martin Eberhard는 Tesla의 설립 과정에서 어떤 역할을 했습니까?...
처리 완료: Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그리고 회...
처리 완료: Tesla가 전기차 시장에서 차지하는 위치와 주요 성과에 대해 설명해 주실 수 있나요?...
처리 완료: Martin Eberhard는 Tesla의 설립 과정에서 어떤 역할을 했습니까?...


Evaluating: 100%|██████████| 15/15 [00:42<00:00,  2.86s/it]


context_recall:
  기본 시스템: 0.667
  개선된 시스템: 0.667
  승자: 기본 시스템 (개선: +0.0%)

faithfulness:
  기본 시스템: 0.727
  개선된 시스템: 0.737
  승자: 개선된 시스템 (개선: +1.3%)

answer_relevancy:
  기본 시스템: 0.675
  개선된 시스템: 0.666
  승자: 기본 시스템 (개선: -1.4%)

context_precision:
  기본 시스템: 0.667
  개선된 시스템: 0.667
  승자: 기본 시스템 (개선: +0.0%)

factual_correctness(mode=f1):
  기본 시스템: 0.543
  개선된 시스템: 0.460
  승자: 기본 시스템 (개선: -15.3%)

